## Informed Search Algorithms

Informed search algorithms use heuristic information to guide the search process towards the goal more efficiently. In this lab, we will implement two popular informed search algorithms: Greedy Best-First Search (GBFS) and A* Search Algorithm. We will apply these algorithms to the Romania problem, a classic search problem in artificial intelligence.

### 1. The Romania Route-Finding Problem

The Romania problem is a classic AI search problem consisting of interconnected cities and road distances. Given a starting city and a destination city, find a path connecting them using an informed search strategy.

**Heuristic Information**: Each city is assigned a Straight-Line Distance (SLD) to Bucharest. The heuristic function:

$$h(n) = \text{Estimated distance from city } n \text{ to the goal}$$

This heuristic helps guide the search toward promising directions.

In [1]:
from heapq import heappush, heappop

# Romania map: weighted graph
graph = {
    'Arad': {'Zerind': 75, 'Sibiu': 140, 'Timisoara': 118},
    'Zerind': {'Arad': 75, 'Oradea': 71},
    'Oradea': {'Zerind': 71, 'Sibiu': 151},
    'Sibiu': {'Arad': 140, 'Oradea': 151, 'Fagaras': 99, 'Rimnicu Vilcea': 80},
    'Timisoara': {'Arad': 118, 'Lugoj': 111},
    'Lugoj': {'Timisoara': 111, 'Mehadia': 70},
    'Mehadia': {'Lugoj': 70, 'Drobeta': 75},
    'Drobeta': {'Mehadia': 75, 'Craiova': 120},
    'Craiova': {'Drobeta': 120, 'Rimnicu Vilcea': 146, 'Pitesti': 138},
    'Rimnicu Vilcea': {'Sibiu': 80, 'Craiova': 146, 'Pitesti': 97},
    'Fagaras': {'Sibiu': 99, 'Bucharest': 211},
    'Pitesti': {'Rimnicu Vilcea': 97, 'Craiova': 138, 'Bucharest': 101},
    'Bucharest': {'Fagaras': 211, 'Pitesti': 101, 'Giurgiu': 90, 'Urziceni': 85},
    'Giurgiu': {'Bucharest': 90},
    'Urziceni': {'Bucharest': 85, 'Hirsova': 98, 'Vaslui': 142},
    'Hirsova': {'Urziceni': 98, 'Eforie': 86},
    'Eforie': {'Hirsova': 86},
    'Vaslui': {'Urziceni': 142, 'Iasi': 92},
    'Iasi': {'Vaslui': 92, 'Neamt': 87},
    'Neamt': {'Iasi': 87}
}

# Straight-line distance to Bucharest
heuristic = {
    'Arad': 366,
    'Bucharest': 0,
    'Craiova': 160,
    'Drobeta': 242,
    'Eforie': 161,
    'Fagaras': 176,
    'Giurgiu': 77,
    'Hirsova': 151,
    'Iasi': 226,
    'Lugoj': 244,
    'Mehadia': 241,
    'Neamt': 234,
    'Oradea': 380,
    'Pitesti': 100,
    'Rimnicu Vilcea': 193,
    'Sibiu': 253,
    'Timisoara': 329,
    'Urziceni': 80,
    'Vaslui': 199,
    'Zerind': 374
}

start = input("Enter the starting city: ").lower().capitalize()
goal = 'Bucharest'

In [2]:
# Validate input
if start not in graph:
    print(f"City '{start}' not found in the Romania map.")
    exit(1)

### 2. Helper Functions

To implement the search algorithms, we will define helper functions for managing the search process, such as maintaining a priority queue for the frontier and tracking visited nodes.

In [3]:
def reconstruct_path(came_from, current):
  total_path = [current]
  while current in came_from:
    current = came_from[current]
    total_path.append(current)
  path = total_path[::-1]
  return path, total_cost(path) # Returns path and its cost

def total_cost(path):
  cost = 0
  for i in range(len(path) - 1):
    cost += graph[path[i]][path[i + 1]]
  return cost # Moved return outside the loop

def display_results(path, cost, expanded_nodes):
  # Check if path is None (no path found)
  if path is None:
    print("{")
    print(f'"path": "No path found", \n"total_cost": \n"N/A", \n"expanded_nodes": {expanded_nodes}')
    print("}")
  else:
    print("{")
    print(f'"path": {list(path)}, \n"total_cost": {cost}, \n"expanded_nodes": {expanded_nodes}')
    print("}")

### 3. Greedy Best-First Search (GBFS)


Greedy Best-First Search always expands the node that appears closest to the goal according to the heuristic value. The Evaluation Function:

$$f(n) = h(n)$$

where:

* $h(n)$ = heuristic estimate to the goal


Advantages

* Fast in many situations
* Explores fewer nodes than uninformed search

Disadvantages

* Does not guarantee the shortest path
* May become trapped in locally optimal choices

Expected Observation: GBFS often reaches the destination quickly but may produce a path that is not optimal.

In [4]:
def greedy_best_first_search(graph, heuristic, start, goal):
  open_set = []
  heappush(open_set, (heuristic[start], start)) # (f_score, node)
  came_from = {}
  closed_set = set() # To keep track of visited/expanded nodes
  expanded_nodes_count = 0

  while open_set:
    f_score, current = heappop(open_set)

    if current in closed_set: # If we've already processed this node, skip
      continue

    expanded_nodes_count += 1 # Increment when a node is truly expanded
    closed_set.add(current) # Mark current node as expanded

    if current == goal:
      path, total_path_cost = reconstruct_path(came_from, current)
      return path, total_path_cost, expanded_nodes_count

    for neighbor, cost in graph[current].items():
      if neighbor not in closed_set: # Only add neighbors not yet expanded
        came_from[neighbor] = current
        heappush(open_set, (heuristic[neighbor], neighbor))

  return None, None, expanded_nodes_count # No path found, return None for path and cost, but expanded_nodes_count

### 4. A* Search Algorithm

A* combines actual travel cost and heuristic information. The Evaluation Function:

$$f(n) = g(n) + h(n)$$

where:

* $g(n)$ = cost from start to current node
* $h(n)$ = estimated cost from current node to goal


Advantages
* Complete
* Optimal (with an admissible heuristic)
* Usually explores fewer nodes than Uniform Cost Search

Disadvantages
* Requires additional memory
* Slightly more computational overhead than GBFS

Expected Observation: A* generally finds the shortest path while maintaining good efficiency.

In [5]:
def a_star_search(graph, heuristic, start, goal):
  open_set = []
  heappush(open_set, (heuristic[start], start)) # (f_score, node) where f_score = h(start) as g(start)=0
  came_from = {}
  g_score = {node: float('inf') for node in graph}
  g_score[start] = 0

  closed_set = set()
  expanded_nodes_count = 0
  while open_set:
    current_f_score_from_heap, current = heappop(open_set)

    if current in closed_set:
        continue

    expanded_nodes_count += 1
    closed_set.add(current)

    if current == goal:
      path, total_path_cost = reconstruct_path(came_from, current)
      return path, total_path_cost, expanded_nodes_count

    for neighbor, cost in graph[current].items():
      tentative_g_score = g_score[current] + cost

      if tentative_g_score < g_score[neighbor]:
        came_from[neighbor] = current
        g_score[neighbor] = tentative_g_score
        f_score = tentative_g_score + heuristic[neighbor]
        heappush(open_set, (f_score, neighbor))

  return None, None, expanded_nodes_count # No path found

### Run and Compare  

We will run both GBFS and A* on the Romania problem and compare their performance in terms of path length, number of nodes expanded, and execution time.

Both algorithsms will return: 
- The path from the start city to the destination city
- The total cost of the path
- The number of nodes expanded during the search. 

We will analyze the results to understand the trade-offs between these two informed search strategies.

In [6]:
# Run both algorithms here
greedy_path, greedy_cost, greedy_expanded = greedy_best_first_search(graph, heuristic, start, goal)
a_star_path, a_star_cost, a_star_expanded = a_star_search(graph, heuristic, start, goal)
print("Greedy Best-First Search:")
display_results(greedy_path, greedy_cost, greedy_expanded)
print("\nA* Search:")
display_results(a_star_path, a_star_cost, a_star_expanded)

Greedy Best-First Search:
{
"path": ['Oradea', 'Sibiu', 'Fagaras', 'Bucharest'], 
"total_cost": 461, 
"expanded_nodes": 4
}

A* Search:
{
"path": ['Oradea', 'Sibiu', 'Rimnicu Vilcea', 'Pitesti', 'Bucharest'], 
"total_cost": 429, 
"expanded_nodes": 6
}
